# MAD（MERT + MAUVE）— Colab GPU

現行データ: 参照 `ref/wav` **99本**、生成 `gen/wav` **100本**（長尺参照は差し替え済み）。

1. リポを `git clone`（スクリプト用）
2. Drive の `testMad/ref/wav` と `gen/wav` を読む（上げ直した現行 WAV）
3. `mad_metric` ＋ transformers ピン（Colab 既存 torch は触らない）

スコアは `-log(MAUVE)`（**低いほど参照に近い**）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# スクリプトは clone、WAV は Drive
REPO = '/content/testMad'
DATA = '/content/drive/MyDrive/testMad'  # Drive 上の wav 置き場
REF_WAV = f'{DATA}/ref/wav'
GEN_WAV = f'{DATA}/gen/wav'

if not os.path.isdir(os.path.join(REPO, 'scripts')):
    !git clone --depth 1 https://github.com/Kakeru0307/testMad.git {REPO}
else:
    print('already cloned', REPO)

assert os.path.isfile(f'{REPO}/scripts/compute_mad.py'), REPO
assert os.path.isdir(REF_WAV), REF_WAV
assert os.path.isdir(GEN_WAV), GEN_WAV
print('ref', len([f for f in os.listdir(REF_WAV) if f.endswith('.wav')]))
print('gen', len([f for f in os.listdir(GEN_WAV) if f.endswith('.wav')]))

In [ ]:
# Colab 既存 torch を残す。MERT 互換のため transformers をピン留め
!pip -q install "mad_metric>=0.0.6" "transformers>=4.40,<4.46"
# ランタイム再起動が求められたら Restart してから続きを実行

In [ ]:
# 参照差し替え後は ref だけ消す。gen_embs が生きていれば CLEAR_REF_ONLY=True / CLEAR_ALL=False
CLEAR_ALL = False
CLEAR_REF_ONLY = True
import shutil
if CLEAR_ALL:
    for _d in (f'{DATA}/cache/gen_embs', f'{DATA}/cache/ref_embs'):
        if os.path.isdir(_d):
            shutil.rmtree(_d); print('cleared', _d)
elif CLEAR_REF_ONLY and os.path.isdir(f'{DATA}/cache/ref_embs'):
    shutil.rmtree(f'{DATA}/cache/ref_embs'); print('cleared ref_embs only')

# batch_size=1（VRAM）。現行参照は最大≈4.9分
!python {REPO}/scripts/compute_mad.py {REF_WAV} {GEN_WAV} \
  --batch-size 1 \
  --ref-embs-dir {DATA}/cache/ref_embs \
  --gen-embs-dir {DATA}/cache/gen_embs \
  --json-out {DATA}/mad_result.json